# Avarodhak – train drone detector (YOLOv8n, 640)
Runtime → Change runtime type → **T4 GPU**, then Run all. Downloads `best.pt` (laptop test), `best.onnx` + `calib.zip` (Hailo compile).

In [ ]:
!pip -q install ultralytics gdown

In [ ]:
%%writefile voc2yolo.py
"""Convert DUT Anti-UAV (Pascal VOC XML) to YOLO format; every object -> class 0 'drone'.
Expects <src>/{train,val,test}/{img,xml}/ and writes <dst>/data.yaml.  Usage: python voc2yolo.py DUT-Anti-UAV dut_yolo"""
import shutil
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

from PIL import Image


def voc_to_yolo(xmin, ymin, xmax, ymax, w, h):
    return (xmin + xmax) / 2 / w, (ymin + ymax) / 2 / h, (xmax - xmin) / w, (ymax - ymin) / h


def convert(src, dst, split):
    out_img, out_lbl = dst / "images" / split, dst / "labels" / split
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    n = 0
    for xml in sorted((src / split / "xml").glob("*.xml")):
        imgs = list((src / split / "img").glob(xml.stem + ".*"))
        if not imgs:
            print("no image for", xml)
            continue
        w, h = Image.open(imgs[0]).size  # real size; some XML <size> tags are wrong
        lines = []
        for obj in ET.parse(xml).getroot().iter("object"):
            b = obj.find("bndbox")
            box = voc_to_yolo(*(float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")), w, h)
            lines.append("0 " + " ".join(f"{v:.6f}" for v in box))
        shutil.copy(imgs[0], out_img / imgs[0].name)
        (out_lbl / f"{xml.stem}.txt").write_text("\n".join(lines))
        n += 1
    print(split, n, "images")


assert voc_to_yolo(10, 20, 30, 60, 100, 200) == (0.2, 0.2, 0.2, 0.2)

if __name__ == "__main__":
    src, dst = Path(sys.argv[1]), Path(sys.argv[2]).resolve()
    splits = [s for s in ("train", "val", "test") if (src / s).exists()]
    for split in splits:
        convert(src, dst, split)
    yaml = [f"path: {dst.as_posix()}"] + [f"{s}: images/{s}" for s in splits] + ["names:", "  0: drone"]
    (dst / "data.yaml").write_text("\n".join(yaml) + "\n")
    print("wrote", dst / "data.yaml")


In [ ]:
# DUT Anti-UAV detection set (train / val / test) from the authors' Google Drive
for fid in ['1RVsSGPUKTdmoyoPTBTWwroyulLek1eTj', '1333uEQfGuqTKslRkkeLSCxylh6AQ0X6n', '1L1zeW1EMDLlXHClSDcCjl3rs_A6sVai0']:
    !gdown -q {fid}
!mkdir -p DUT-Anti-UAV && for z in train val test; do unzip -q -o $z.zip -d DUT-Anti-UAV; done
!python voc2yolo.py DUT-Anti-UAV dut_yolo

In [ ]:
# Checkpoints go to Drive so a Colab disconnect doesn't lose the run: just Run all again and it resumes.
from google.colab import drive; drive.mount('/content/drive')
import os
from ultralytics import YOLO
last = '/content/drive/MyDrive/avarodhak/drone/weights/last.pt'
if os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO('yolov8n.pt').train(data='dut_yolo/data.yaml', imgsz=640, epochs=100, batch=32, patience=20,
                             project='/content/drive/MyDrive/avarodhak', name='drone', exist_ok=True)

In [ ]:
!yolo detect val model=/content/drive/MyDrive/avarodhak/drone/weights/best.pt data=dut_yolo/data.yaml split=test imgsz=640

In [ ]:
# opset 11 ONNX is what the Hailo Model Zoo yolov8n recipe expects
!yolo export model=/content/drive/MyDrive/avarodhak/drone/weights/best.pt format=onnx imgsz=640 opset=11
!mkdir -p calib && ls dut_yolo/images/train/*.jpg | shuf -n 1024 | xargs -I{} cp {} calib/
!cd calib && zip -q ../calib.zip *.jpg
from google.colab import files
for f in ['/content/drive/MyDrive/avarodhak/drone/weights/best.pt', '/content/drive/MyDrive/avarodhak/drone/weights/best.onnx', 'calib.zip']:
    files.download(f)